# Recreating ROME: Locating and Editing Factual Associations in GPT

This notebook recreates the core experiments from:
> **Meng et al. (2022)** — *Locating and Editing Factual Associations in GPT* (NeurIPS 2022)

The paper has two main contributions:
1. **Causal Tracing** : identifying *where* facts are stored (middle-layer MLP modules at the last subject token)
2. **ROME** : a rank-one weight edit that inserts a new fact with both generalization and specificity

We use the **Murano Pipeline API** throughout.

---
### Notebook structure
| Section | Paper experiment |
|---------|------------------|
| 1 | Setup & model loading |
| 2 | **Causal Tracing** — record clean vs. corrupted activations, measure indirect effect |
| 3 | **Steering vector as a proxy for factual direction** — validate MLP early-site hypothesis |
| 4 | **Ablation** — remove the factual direction, observe prediction degradation |
| 5 | **Generation comparison** — clean vs. steered/ablated outputs |

---
## 1. Imports

In [2]:
import torch

from murano import MuranoModel, Pipeline
from murano.dataset import MuranoDataset
from murano.steps.load import Load
from murano.steps.record import Record, ActivationStore
from murano.steps.train import SteeringVector, SteeringResult
from murano.steps.intervene import Intervene, steer_direction, ablate_direction
from murano.results import Results
from murano.plotting.plotly_utils import plot_heatmap, plot_line_chart

---
## 2. Model

In [3]:
MODEL_ID = "gpt2"  # swap to "gpt2-xl" when compute allows

model = MuranoModel(MODEL_ID)

print(f"Model   : {MODEL_ID}")
print(f"d_model : {model.d_model}")
print(f"n_layers: {model.n_layers}")

Model   : gpt2
d_model : 768
n_layers: 12


---
## 3. Data

In [4]:
FACTUAL_PROMPTS = [
    "The Space Needle is located in the city of",
    "The Eiffel Tower is located in the city of",
    "The Colosseum is located in the city of",
    "The Louvre is located in the city of",
    "The Sagrada Familia is located in the city of",
    "The Big Ben is located in the city of",
    "The Kremlin is located in the city of",
    "The Acropolis is located in the city of",
]

# Expected first-token answers for factual_recall_rate()
EXPECTED_CITIES = [
    "Seattle",
    "Paris",
    "Rome",
    "Paris",
    "Barcelona",
    "London",
    "Moscow",
    "Athens",
]

clean_dataset = MuranoDataset(positive_texts=FACTUAL_PROMPTS, negative_texts=[])

---
## 4. Helpers

In [12]:
NOISE_SCALE = 3.0  # ν = 3σ_t, matching the paper


def record_with_noise(model, dataset, **record_kwargs) -> ActivationStore:
    """Run Record with Gaussian noise injected at the embedding level.

    Uses register_forward_pre_hook on layer 0 so the noise is added to
    the hidden states BEFORE block 0 processes them — equivalent to
    corrupting the token embeddings as done in the paper.
    """

    def _pre_hook(module, args):
        h = args[0]
        # randn_like inherits h's dtype automatically — no cast needed here
        noised = h + torch.randn_like(h) * h.std() * NOISE_SCALE
        return (noised, *args[1:])

    handle = model.layer(0).register_forward_pre_hook(_pre_hook)
    try:
        results = Pipeline([Load(dataset), Record(model, **record_kwargs)]).run()
        return results["record"]
    finally:
        handle.remove()


def factual_recall_rate(model, prompts, expected, top_k=10):
    """Fraction of prompts where the correct city is in the model's top-k
    predictions at the last token position — a direct proxy for the paper's
    P[o] metric.
    """
    tokenizer = model.tokenizer
    lm = model._lm
    device = next(lm.parameters()).device  # wherever the model actually lives
    hits = 0
    for prompt, city in zip(prompts, expected):
        inputs = tokenizer(prompt, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}  # ← the fix
        with torch.no_grad():
            logits = lm(**inputs).logits[0, -1]
        city_id = tokenizer(" " + city, add_special_tokens=False).input_ids[0]
        rank = (logits > logits[city_id]).sum().item()
        if rank < top_k:
            hits += 1
    return hits / len(prompts)


def factual_recall_rate_with_hook(
    model, prompts, expected, hook_fn, layer_idx=0, top_k=10
):
    """Same as factual_recall_rate but with a pre-hook active on layer_idx."""
    handle = model.layer(layer_idx).register_forward_pre_hook(hook_fn)
    try:
        return factual_recall_rate(model, prompts, expected, top_k=top_k)
    finally:
        handle.remove()


print("Helpers defined.")

Helpers defined.


---
## 5. Record clean activations

In [6]:
clean_results = Pipeline(
    [
        Load(clean_dataset),
        Record(
            model,
            layers="all",
            modules=["residual", "mlp", "attn"],
            position="last",
            batch_size=4,
        ),
    ]
).run()

clean_activations = clean_results["record"]
print("Clean recording done.")
print("Sample key:", list(clean_activations.positive.keys())[0])

Clean recording done.
Sample key: (0, 'residual')


---
## 6. Record corrupted activations

The pre-hook fires before block 0 runs, so every layer sees the noised input —
equivalent to corrupting the token embeddings as the paper does.

In [7]:
corrupt_activations = record_with_noise(
    model,
    clean_dataset,
    layers="all",
    modules=["residual", "mlp", "attn"],
    position="last",
    batch_size=4,
)

# positive = clean (subject present), negative = corrupted (subject absent)
causal_store = ActivationStore(
    positive=clean_activations.positive,
    negative=corrupt_activations.positive,
)

print("Corrupted recording done.")

Corrupted recording done.


---
## 7. Compute steering directions

In [ ]:
r = Results()
r["record"] = causal_store
steering: SteeringResult = SteeringVector(normalize=True)(r)["steering"]

all_layers = sorted(set(k[0] for k in steering.separation_scores.keys()))
modules = ["residual", "mlp", "attn"]
scores = steering.separation_scores

print(f"Best key overall : {steering.best_layer}")
print(f"Layers           : {all_layers}")
print()
print("Per-module separation scores:")
print(f"{'Layer':<6}", "  ".join(f"{m:>10}" for m in modules))
for layer in all_layers:
    row = "  ".join(f"{scores.get((layer, m), 0.0):>10.4f}" for m in modules)
    print(f"{layer:<6}  {row}")

Best key overall : (2, 'mlp')
Layers           : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

Per-module separation scores:
Layer    residual         mlp        attn
0           7.1347      5.6021      8.1856
1           6.5976     12.6888     11.0863
2          16.6809     22.5237      6.9959
3          15.1817     11.0711      4.7789
4          14.1290      6.7501      3.4164
5          13.2371      4.0981      2.0819
6          12.3671      3.2014      1.4254
7          11.9182      1.7243      1.4168
8          11.3330      3.0850      1.4350
9          11.1967      5.7279      1.4570
10         10.0083      2.7918      1.3839
11          1.8337      0.9151     13.6114


---
## 8. Separation score per layer — Figure 2 replica

Expected: **MLP peaks in middle layers** (early site); **attention peaks late** (late site).

In [ ]:
fig = plot_line_chart(
    x_data=all_layers,
    y_series={
        mod: [scores.get((layer, mod), 0.0) for layer in all_layers] for mod in modules
    },
    title="Separation Score per Layer (proxy for Average Indirect Effect — Fig. 2)",
    x_label="Layer",
    y_label="Separation Score",
)
fig.show()

---
## 9. Token position comparison

Expected: **`last` token peaks highest at middle layers** for MLP.

In [ ]:
# Bug 2 fix: use record_with_noise for all positions — no Intervene→Record chain
position_scores = {}

for position in ["last", "first", "mean"]:
    clean_r = Pipeline(
        [
            Load(clean_dataset),
            Record(model, layers="all", modules="mlp", position=position, batch_size=4),
        ]
    ).run()["record"]

    corrupt_r = record_with_noise(
        model,
        clean_dataset,
        layers="all",
        modules="mlp",
        position=position,
        batch_size=4,
    )

    store = ActivationStore(positive=clean_r.positive, negative=corrupt_r.positive)
    r2 = Results()
    r2["record"] = store
    pos_steering = SteeringVector(normalize=True)(r2)["steering"]

    position_scores[position] = pos_steering.separation_scores
    print(f"position='{position}' best layer: {pos_steering.best_layer}")

fig = plot_line_chart(
    x_data=all_layers,
    y_series={
        f"MLP @ position='{pos}'": [
            position_scores[pos].get(layer, 0.0) for layer in all_layers
        ]
        for pos in ["last", "first", "mean"]
    },
    title="MLP Separation Score by Token Position",
    x_label="Layer",
    y_label="Separation Score",
)
fig.show()

position='last' best layer: 2
position='first' best layer: 0
position='mean' best layer: 0


---
## 10. Ablation — factual recall rate drop

Check whether the correct city is in the
model's top-10 predictions, using token probabilities directly.
A drop after ablation confirms the direction is genuinely factual.

In [ ]:
mlp_directions = {
    layer: steering.direction_per_layer[(layer, "mlp")]
    for layer in all_layers
    if (layer, "mlp") in steering.direction_per_layer
}

best_mlp_layer = max(
    mlp_directions,
    key=lambda layer: scores.get((layer, "mlp"), 0.0),
)
print(f"Best MLP layer: {best_mlp_layer}")

ablate_fn = ablate_direction(mlp_directions)

# Baseline: no intervention
clean_rate = factual_recall_rate(model, FACTUAL_PROMPTS, EXPECTED_CITIES, top_k=10)
print(f"Clean recall (top-10)  : {clean_rate:.2%}")


def _ablate_pre_hook(module, args):
    h = args[0]
    direction = mlp_directions.get(best_mlp_layer)
    if direction is None:
        return args
    # Cast direction to match the hidden-state dtype AND device
    d = (direction / direction.norm()).to(dtype=h.dtype, device=h.device)
    h = h - (h @ d).unsqueeze(-1) * d
    return (h, *args[1:])


ablated_rate = factual_recall_rate_with_hook(
    model,
    FACTUAL_PROMPTS,
    EXPECTED_CITIES,
    hook_fn=_ablate_pre_hook,
    layer_idx=best_mlp_layer,
    top_k=10,
)
print(f"Ablated recall (top-10): {ablated_rate:.2%}")
print("Expected: ablated should be lower than clean.")

Best MLP layer: 2
Clean recall (top-10)  : 75.00%
Ablated recall (top-10): 62.50%
Expected: ablated should be lower than clean.


---
## 11. Generation comparison — clean vs steered vs ablated

In [17]:
EVAL_PROMPTS = [
    "The Space Needle is located in the city of",
    "The Eiffel Tower is located in the city of",
    "The Colosseum is located in the city of",
]
eval_small = MuranoDataset(positive_texts=EVAL_PROMPTS, negative_texts=[])
steer_fn = steer_direction(mlp_directions, alpha=6.0)

steer_result = Pipeline(
    [
        Load(eval_small),
        Intervene(
            model,
            fn=steer_fn,
            modules="mlp",
            layers=[best_mlp_layer],
            gen_kwargs={"max_new_tokens": 10},
        ),
    ]
).run()["intervene"]

ablate_result = Pipeline(
    [
        Load(eval_small),
        Intervene(
            model,
            fn=ablate_fn,
            modules="mlp",
            layers=[best_mlp_layer],
            gen_kwargs={"max_new_tokens": 10},
        ),
    ]
).run()["intervene"]

col_w = 35
header = (
    f"{'Prompt':<45} | {'Clean':^{col_w}} | {'Steered':^{col_w}} | {'Ablated':^{col_w}}"
)
print(header)
print("-" * len(header))

for i, prompt in enumerate(EVAL_PROMPTS):
    clean = steer_result.clean_generations[i].strip().replace("\n", " ")
    steered = steer_result.modified_generations[i].strip().replace("\n", " ")
    ablated = ablate_result.modified_generations[i].strip().replace("\n", " ")

    # Truncate each column to col_w chars so nothing overflows
    clean = (clean[: col_w - 3] + "...") if len(clean) > col_w else clean
    steered = (steered[: col_w - 3] + "...") if len(steered) > col_w else steered
    ablated = (ablated[: col_w - 3] + "...") if len(ablated) > col_w else ablated

    print(f"{prompt:<45} | {clean:^{col_w}} | {steered:^{col_w}} | {ablated:^{col_w}}")

Intervene:   0%|          | 0/3 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Intervene:  33%|███▎      | 1/3 [00:00<00:00,  3.59it/s]Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Intervene:  67%|██████▋   | 2/3 [00:00<00:00,  4.08it/s]Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Intervene:   0%|          | 0/3 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Intervene:  33%|███▎      | 1/3 [00:00<00:00,  4.17it/s]Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Intervene:  67%|

Prompt                                        |                Clean                |               Steered               |               Ablated              
---------------------------------------------------------------------------------------------------------------------------------------------------------------
The Space Needle is located in the city of    |  Tuscany, Italy. It is located in   |  Tarkir, in the center of the city  |  Tarkir, in the north-east corner  
The Eiffel Tower is located in the city of    |     Paris, France.  The Eiffel      |     Paris, France.  The Eiffel      | Paris, and is the tallest buildi...
The Colosseum is located in the city of       | Tuscany, Italy, and is the largest  |      Tuscany, Italy.  The Col       |  Tuscany, Italy. It is the largest 


---
## 12. Layer sweep

In [ ]:
sweep_layers = list(range(0, model.n_layers, 3))
recall_rates = {}

for layer in sweep_layers:
    direction = mlp_directions.get(layer)
    if direction is None:
        recall_rates[layer] = factual_recall_rate(
            model, FACTUAL_PROMPTS, EXPECTED_CITIES
        )
        continue

    def _hook(module, args, _d=direction):
        h = args[0]
        # Cast to match hidden-state dtype — same fix as _ablate_pre_hook
        d = (_d / _d.norm()).to(dtype=h.dtype, device=h.device)
        h = h - (h @ d).unsqueeze(-1) * d
        return (h, *args[1:])

    rate = factual_recall_rate_with_hook(
        model,
        FACTUAL_PROMPTS,
        EXPECTED_CITIES,
        hook_fn=_hook,
        layer_idx=layer,
        top_k=10,
    )
    recall_rates[layer] = rate
    print(f"  Layer {layer:2d}: recall = {rate:.2%}")

fig = plot_line_chart(
    x_data=sweep_layers,
    y_series={
        "Recall after ablation (lower = more decisive)": [
            recall_rates[layer] for layer in sweep_layers
        ]
    },
    title="Layer Sweep: Factual Recall after MLP Ablation (replicates Fig. 5)",
    x_label="Ablated layer",
    y_label="Recall rate (top-10)",
)
fig.show()

  Layer  0: recall = 75.00%
  Layer  3: recall = 0.00%
  Layer  6: recall = 0.00%
  Layer  9: recall = 62.50%


---
## 13. Multi-module heatmap — Figure 2b/c replica

In [ ]:
z_data = [[scores.get((layer, mod), 0.0) for layer in all_layers] for mod in modules]

fig = plot_heatmap(
    z_data=z_data,
    x_labels=[str(layer) for layer in all_layers],
    y_labels=modules,
    title="Separation Score per Module and Layer (replicates Fig. 2)",
    color_scale="Viridis",
)
fig.show()
# Expected: mlp row bright in middle layers; attn row bright near the end